<a href="https://colab.research.google.com/github/ababacaryoro/nlp-course/blob/main/assignments/1/assignment1_part1_regex.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 1 – Part 1: regular expressions and date extraction

**Points:** 10 (half of Assignment 1)

**Rules**
1. Complete every function marked `# YOUR CODE HERE`.
2. Do **not** change the function names or their signatures, and return exactly the type stated.
3. Test each function with the cell that follows it.
4. Work alone. Using an AI assistant to write the answers is not allowed.

## The data

`data/dates.txt` contains 500 lines of medical notes. Each line contains **one date**, written in one of many formats:

- `04/20/2009; 04/20/09; 4/20/09; 4/3/09`
- `Mar-20-2009; Mar 20, 2009; March 20, 2009; Mar. 20, 2009; Mar 20 2009`
- `20 Mar 2009; 20 March 2009; 20 Mar. 2009; 20 March, 2009`
- `Mar 20th, 2009; Mar 21st, 2009; Mar 22nd, 2009`
- `Feb 2009; Sep 2009; Oct 2010`
- `6/2008; 12/2009`
- `2009; 2010`

## Setup

In [47]:
# Google Colab only: install the libraries that Colab does not ship (about 1 minute).
# Uncomment the next line and run the cell. Nothing to do for a local installation made with `uv sync`.
# !pip install -q -r https://raw.githubusercontent.com/ababacaryoro/nlp-course/main/requirements.txt

In [48]:
import os, re, urllib.request
import pandas as pd
import numpy as np
from datetime import datetime

def local_file(name):
    path = os.path.join("data", name)
    if not os.path.exists(path):
        os.makedirs("data", exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/ababacaryoro/nlp-course/main/assignments/1/data/{name}", path)
    return path

with open(local_file("dates.txt"), encoding="utf-8") as f:
    df = pd.Series([line.strip() for line in f])
print(f"{len(df)} notes")
df.head()

500 notes


,0
0,03/25/93 Total time of visit (in minutes):
1,6/18/85 Primary Care Doctor:
2,sshe plans to move as of 7/8/71 In-Home Servic...
3,7 on 9/27/75 Audit C Score Current:
4,2/6/96 sleep studyPain Treatment Pain Level (N...


## Question 1 (1 point)

Return the list of every date written as `MM/DD/YY` or `MM/DD/YYYY` (for example `03/25/93`, `6/18/85`, `5/24/1990`) found in the notes.

In [49]:
import re

def question_one():
    # A straightforward way to find MM/DD/YY or MM/DD/YYYY
    # We look for 1 or 2 digits, a slash or dash, 1 or 2 digits, another slash or dash, then 2 or 4 digits.
    matches = []
    simple_pattern = r'\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b'
    for line in df:
        for item in re.findall(simple_pattern, line):
            matches.append(item)
    return matches

q1 = question_one()
print(f"{len(q1)} dates, first 10: {q1[:10]}")

125 dates, first 10: ['03/25/93', '6/18/85', '7/8/71', '9/27/75', '2/6/96', '7/06/79', '5/18/78', '10/24/89', '3/7/86', '4/10/71']


## Question 2 (1 point)

Return the list of every date that starts with a month name followed by a day and a year: `Mar-20-2009`, `Mar 20, 2009`, `March 20, 2009`, `Mar. 20, 2009`, `Mar 20 2009`, `Mar 20th, 2009`.

In [50]:
def question_two():
    # Find month name followed by day and year (e.g., Mar-20-2009, Mar 20, 2009)
    months_regex = r'(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*'
    pattern = months_regex + r'[-,.\s]+\d{1,2}(?:st|nd|rd|th)?[-,.\s]+\d{4}'

    all_found = []
    for line in df:
        found_items = re.findall(pattern, line, flags=re.IGNORECASE)
        for item in found_items:
            all_found.append(item)
    return all_found

q2 = question_two()
print(f"{len(q2)} dates, first 10: {q2[:10]}")

34 dates, first 10: ['April 11, 1990', 'May 30, 2001', 'Feb 18, 1994', 'February 18, 1981', 'October. 11, 2013', 'Jan 24 1986', 'July 26, 1978', 'December 23, 1999', 'May 15, 1989', 'September 06, 1995']


## Question 3 (1 point)

Return the list of every date written as `DD Month YYYY`: `20 Mar 2009`, `20 March 2009`, `20 Mar. 2009`, `20 March, 2009`.

In [51]:
def question_three():
    # Find 'DD Month YYYY' (e.g. 20 Mar 2009, 20 March 2009)
    months = r'(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*'
    pattern = r'\b\d{1,2}(?:st|nd|rd|th)?\s+' + months + r'[.,]?\s+\d{4}\b'

    results = []
    for line in df:
        for match in re.findall(pattern, line, flags=re.IGNORECASE):
            results.append(match)
    return results

q3 = question_three()
print(f"{len(q3)} dates, first 10: {q3[:10]}")

69 dates, first 10: ['24 Jan 2001', '10 Sep 2004', '26 May 1982', '28 June 2002', '06 May 1972', '25 Oct 1987', '14 Oct 1996', '30 Nov 2007', '28 June 1994', '14 Jan 1981']


## Question 4 (1 point)

`question_four(text)` returns the list of e-mail addresses found in `text`.

In [52]:
def question_four(text):
    # Standard pattern for finding email addresses
    email_regex = r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'
    raw_emails = re.findall(email_regex, text)
    # Filter to make sure email ends with at least a 2 letter domain extension
    clean_emails = []
    for email in raw_emails:
        if len(email.split('.')[-1]) >= 2:
            clean_emails.append(email)
    return clean_emails

test_text = """
Contact us at support@company.com or sales@company.org.
You can also reach john.doe@email.co.uk or jane_doe123@university.edu.
Invalid: @invalid.com, user@, not-an-email
"""
print(question_four(test_text))

['support@company.com', 'john.doe@email.co.uk']


## Question 5 (1 point)

`question_five(text)` cleans a text: remove digits, remove punctuation, lowercase, normalize whitespace.

In [53]:
def question_five(text):
    # Strip out digits
    no_digits = re.sub(r'\d', '', text)
    # Strip out punctuation characters
    no_punctuation = re.sub(r'[^\w\s]', ' ', no_digits)
    # Lowercase everything
    lowered = no_punctuation.lower()
    # Split by whitespace and rejoin with single space
    words = lowered.split()
    return ' '.join(words)

## Question 6 (2 points)

`question_six(text)` extracts the phone numbers written as `XXX-XXX-XXXX`, `(XXX) XXX-XXXX`, `XXX.XXX.XXXX` or `XXX XXX XXXX` and returns them normalized as `XXX-XXX-XXXX`. Sequences such as `12-34-5678` or `1234567890` are not valid.

In [54]:
import re

def question_six(text):
    # Match phone numbers with optional parentheses around the area code and common separators
    phone_pattern = r'\(?\d{3}\)?[\s.-]?\d{3}[\s.-]?\d{4}'
    raw_matches = re.findall(phone_pattern, text)

    normalized_numbers = []
    for match in raw_matches:
        # Strip everything except actual digits
        only_digits = "".join(char for char in match if char.isdigit())
        if len(only_digits) == 10:
            formatted = f"{only_digits[:3]}-{only_digits[3:6]}-{only_digits[6:]}"
            normalized_numbers.append(formatted)

    return normalized_numbers

test_text = """
Call us at 123-456-7890 or (555) 123-4567.
You can also reach us at 888.555.1234 or 999 888 7777.
Invalid: 12-34-5678, 1234567890
"""

print(question_six(test_text))

['123-456-7890', '555-123-4567', '888-555-1234', '999-888-7777', '123-456-7890']


## Question 7 (3 points)

Extract the date of every note and sort the notes chronologically.

**Rules**
- `xx/xx/xx` is `mm/dd/yy`.
- A 2-digit year is in the 1900s (`1/5/89` is January 5th, 1989).
- When the day is missing (`9/2009`, `Feb 2009`), use the first day of the month.
- When the month is missing (`2010`), use January 1st.

`question_seven()` returns a `pandas.Series` of length 500 whose **values are the original indices** of the notes, in ascending chronological order.

Example: if the notes were `0: "1999"`, `1: "2010"`, `2: "1978"`, the function returns the Series `[2, 0, 1]`.

In [55]:
def question_seven():
    dates_list = []

    # Month mapper dictionary
    month_lookup = {
        'jan': 1, 'feb': 2, 'mar': 3, 'apr': 4, 'may': 5, 'jun': 6,
        'jul': 7, 'aug': 8, 'sep': 9, 'oct': 10, 'nov': 11, 'dec': 12
    }

    # Defined search regexes
    re_type1 = r'(\d{1,2})[/-](\d{1,2})[/-](\d{2,4})' # MM/DD/YY or MM/DD/YYYY
    re_type2 = r'((?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*)[-,.\s]+(\d{1,2})[a-z]*[-,.\s]+(\d{4})' # Month DD, YYYY
    re_type3 = r'(\d{1,2})[a-z]*\s+((?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*)[.,]?\s+(\d{4})' # DD Month YYYY
    re_type4 = r'((?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*)[.,]?\s+(\d{4})' # Month YYYY
    re_type5 = r'(\d{1,2})/(\d{4})' # MM/YYYY
    re_type6 = r'\b(\d{4})\b' # YYYY

    for idx, text in enumerate(df):
        # 1. Check MM/DD/YY or MM/DD/YYYY
        match = re.search(re_type1, text)
        if match:
            month = int(match.group(1))
            day = int(match.group(2))
            year = int(match.group(3))
            if year < 100:
                year += 1900
            dates_list.append((idx, datetime(year, month, day)))
            continue

        # 2. Check Month DD, YYYY
        match = re.search(re_type2, text, re.IGNORECASE)
        if match:
            month_str = match.group(1)[:3].lower()
            month = month_lookup[month_str]
            day = int(match.group(2))
            year = int(match.group(3))
            dates_list.append((idx, datetime(year, month, day)))
            continue

        # 3. Check DD Month YYYY
        match = re.search(re_type3, text, re.IGNORECASE)
        if match:
            day = int(match.group(1))
            month_str = match.group(2)[:3].lower()
            month = month_lookup[month_str]
            year = int(match.group(3))
            dates_list.append((idx, datetime(year, month, day)))
            continue

        # 4. Check Month YYYY
        match = re.search(re_type4, text, re.IGNORECASE)
        if match:
            month_str = match.group(1)[:3].lower()
            month = month_lookup[month_str]
            year = int(match.group(2))
            dates_list.append((idx, datetime(year, month, 1)))
            continue

        # 5. Check MM/YYYY
        match = re.search(re_type5, text)
        if match:
            month = int(match.group(1))
            year = int(match.group(2))
            dates_list.append((idx, datetime(year, month, 1)))
            continue

        # 6. Check YYYY
        match = re.search(re_type6, text)
        if match:
            year = int(match.group(1))
            dates_list.append((idx, datetime(year, 1, 1)))
            continue

        # Fallback
        dates_list.append((idx, datetime(1900, 1, 1)))

    # Sort chronologically, then return original indices
    dates_list.sort(key=lambda item: item[1])
    sorted_indices = [item[0] for item in dates_list]
    return pd.Series(sorted_indices)

q7 = question_seven()
print(f"Length: {len(q7)}, first 10 indices: {list(q7.head(10))}")


Length: 500, first 10 indices: [466, 470, 481, 493, 9, 84, 2, 53, 28, 474]


## Check before submitting

In [56]:
checks = [("question_one", lambda: question_one(), list), ("question_two", lambda: question_two(), list),
          ("question_three", lambda: question_three(), list), ("question_four", lambda: question_four("a@b.com"), list),
          ("question_five", lambda: question_five("Hello 1"), str), ("question_six", lambda: question_six("123-456-7890"), list),
          ("question_seven", lambda: question_seven(), pd.Series)]
for name, call, typ in checks:
    try:
        r = call()
        assert isinstance(r, typ), f"should return {typ.__name__}"
        if name == "question_seven":
            assert len(r) == 500, "should have length 500"
        print(f"OK   {name}")
    except Exception as e:
        print(f"FAIL {name}: {e}")

OK   question_one
OK   question_two
OK   question_three
OK   question_four
OK   question_five
OK   question_six
OK   question_seven


---
## Submission

1. Run the whole notebook (`Kernel > Restart & Run All`): the last cell checks that every function exists and returns the right type.
2. Commit the notebook with its outputs in `assignments/1/` of your course repository and push.
3. Send an e-mail to the teacher to say that the assignment is ready (the repository link is only needed in your first e-mail).

Grading is automatic: the functions are imported and compared with reference results. Do not rename them and do not change their signatures.